In [ ]:
import pandas as pd

df = pd.read_csv(r"../data/raw/apartments_pl_2023_08.csv")

df

In [1]:
nulos = df.isnull().sum().sort_values(ascending=False)
print(nulos)

NameError: name 'df' is not defined

In [ ]:
size = df.shape
print(size)

In [ ]:
duplas = df.duplicated().sum()
print(duplas)

In [ ]:
colunas = df.columns
print(colunas)

In [ ]:
tipo = df.info()

In [ ]:
df.describe()

In [ ]:
df['price_per_m2'] = df['price']/df['squareMeters']
df['price_per_m2'] = df['price_per_m2'].round(2)
df

In [ ]:
mediana = df.groupby('city')['price_per_m2'].median()
mediana = mediana.sort_values(ascending=False)
mediana

In [ ]:
import os

lista = []
for i in os.listdir('../data/raw'):
    if 'apartments_pl' in i:
        lista.append(i)

dfs =[]
for x in lista:
    arq = pd.read_csv('../data/raw/'+x)
    index1 = x.index('l')+2
    index2 = x.index('.')
    month = x[index1 : index2]
    arq['month'] = month
    dfs.append(arq)
ap = pd.concat(dfs)

In [ ]:
ap.shape

In [ ]:
ap['month'].value_counts()

In [ ]:
ap_sorted_month = ap.sort_values(by='month')
ap_nd = ap_sorted_month.drop_duplicates(subset='id',keep='last')

ap_nd.shape

In [ ]:
ap_nd = ap_nd.drop(columns='condition')
ap_nd.shape

In [ ]:
ap_nd['rooms'].isnull().sum().dtype

In [ ]:
ap_nd['floor'].isnull().sum()

In [ ]:
ap_nd['rooms'] = ap_nd['rooms'].astype('Int64')

In [ ]:
ap_nd['floor'] = ap_nd['floor'].astype('Int64')

In [ ]:
ap_nd[['rooms','floor']].dtypes

In [ ]:
ap_nd['price_per_m2'] = ap_nd['price'] / ap_nd['squareMeters']
ap_nd['price_per_m2'] = ap_nd['price_per_m2'].round(2)
ap_nd['price_per_m2']

In [ ]:
ap_nd['price_per_m2'].describe()

In [ ]:
ap_nd['city_median_m2'] = ap_nd.groupby('city')['price_per_m2'].transform('median')

ap_nd

In [ ]:
ap_nd['ratio'] = ap_nd['price_per_m2'] / ap_nd['city_median_m2']
ap_nd['ratio'].describe()

In [ ]:
import numpy as np

ap_nd['is_outlier'] = np.where((ap_nd['ratio'] < 0.5) | (ap_nd['ratio'] > 2.0) , True, False)

ap_nd['is_outlier'].sum()

In [ ]:
ap_nd[ap_nd['is_outlier']]['city'].value_counts()

In [ ]:
x = ap_nd.groupby('city')['is_outlier'].mean().sort_values(ascending=False)
x

In [ ]:
ap_nd.to_csv('../data/clean/apartments_clean.csv', index=False)

In [ ]:
ap_nd.shape

In [36]:
removed_cols = set(ap.columns) - set(ap_nd.columns)
int_cols = list(ap_nd.columns[ap_nd.dtypes == 'Int64'])
n_out = ap_nd['is_outlier'].sum()

print('=== DATA QUALITY REPORT ===')
print(f'Rows in raw data:      {len(ap):,}')
print(f'Duplicates removed:    {len(ap) - len(ap_nd):,}')
print(f'Rows in clean data:    {len(ap_nd):,}')
print(f'Columns removed:       {sorted(removed_cols)}')
print(f'Converted to Int64:    {int_cols}')
print(f'Outliers flagged:      {n_out:,} ({n_out / len(ap_nd):.2%})')

=== DATA QUALITY REPORT ===
Rows in raw data:      195,568
Duplicates removed:    102,601
Rows in clean data:    92,967
Columns removed:       ['condition']
Converted to Int64:    ['rooms', 'floor']
Outliers flagged:      426 (0.46%)
